# Task: Water Meter Data Loading and Exploration
#### in Phase 2 - Module 1: Data Fundamentals

======================================================================================================

Load CSV file with water meter data

In [57]:
import pandas as pd
wwdata = pd.read_csv("..\\water_meter_readings.csv")
print("Successfully loaded water meter readings data")
wwdata.head(10)

Successfully loaded water meter readings data


,DateTime,Meter_ID,Flow_Rate,Pressure,Temperature,Water_Volume
0,2024-01-01 00:00:00,METER_001,1.23,2.85,15.2,450
1,2024-01-01 01:00:00,METER_002,0.87,2.45,14.8,320
2,2024-01-01 02:00:00,METER_003,1.56,3.12,15.5,520
3,2024-01-01 03:00:00,METER_004,0.45,2.15,14.2,180
4,2024-01-01 04:00:00,METER_005,1.34,2.78,15.1,420
5,2024-01-01 05:00:00,METER_001,1.45,2.92,15.3,480
6,2024-01-01 06:00:00,METER_002,0.92,2.52,14.9,350
7,2024-01-01 07:00:00,METER_003,1.78,3.25,15.7,580
8,2024-01-01 08:00:00,METER_004,0.56,2.28,14.5,210
9,2024-01-01 09:00:00,METER_005,1.67,2.95,15.4,530


Display sensor information

In [58]:
wwdata.info()

<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   DateTime      55 non-null     str    
 1   Meter_ID      55 non-null     str    
 2   Flow_Rate     55 non-null     float64
 3   Pressure      55 non-null     float64
 4   Temperature   55 non-null     float64
 5   Water_Volume  55 non-null     int64  
dtypes: float64(3), int64(1), str(2)
memory usage: 2.7 KB


Check data types and missing values

In [59]:
print("Data Types:")
print(wwdata.dtypes)
print("*******************************")
print("Missing Values:")
print(wwdata.isnull().sum())

Data Types:
DateTime            str
Meter_ID            str
Flow_Rate       float64
Pressure        float64
Temperature     float64
Water_Volume      int64
dtype: object
*******************************
Missing Values:
DateTime        0
Meter_ID        0
Flow_Rate       0
Pressure        0
Temperature     0
Water_Volume    0
dtype: int64


Identify any duplicate readings

In [60]:
duplicates = wwdata[wwdata.duplicated()]
print("Number of duplicate rows:", len(duplicates))
duplicates

Number of duplicate rows: 0


,DateTime,Meter_ID,Flow_Rate,Pressure,Temperature,Water_Volume


Identify outliers using statistical methods

In [61]:
numeric_data = wwdata[["Flow_Rate", "Pressure", "Water_Volume"]]

Q1 = numeric_data.quantile(0.25)
Q3 = numeric_data.quantile(0.75)

IQR = Q3 - Q1

outlier_check = ((numeric_data < (Q1 - 1.5 * IQR)) |
                 (numeric_data > (Q3 + 1.5 * IQR)))

outlier_values = numeric_data[outlier_check]
display(outlier_values.dropna(how="all"))

outliers = wwdata[outlier_check.any(axis=1)]

outliers

,Flow_Rate,Pressure,Water_Volume
30,4.67,1.25,NaN
35,5.12,0.95,NaN
40,4.89,1.05,NaN


,DateTime,Meter_ID,Flow_Rate,Pressure,Temperature,Water_Volume
30,2024-01-02 06:00:00,METER_001,4.67,1.25,16.8,950
35,2024-01-02 11:00:00,METER_001,5.12,0.95,17.2,1020
40,2024-01-02 16:00:00,METER_001,4.89,1.05,17.1,980


### Questions & Answers

1. Which meter recorded the highest flow rate in the dataset?

In [62]:
wwdata.Meter_ID.loc[wwdata.Flow_Rate.idxmax()]

'METER_001'

2. What is the average water temperature for METER_003?

In [63]:
wwdata.Temperature[wwdata.Meter_ID == "METER_003"].mean()

np.float64(15.78181818181818)

3. How much total water volume was recorded by METER_005 on 2024-01-01?

In [64]:
wwdata[(wwdata.Meter_ID == "METER_005") & (wwdata.DateTime.str.startswith("2024-01-01"))].Water_Volume.sum()

np.int64(2230)

4. Filter data by meter

    (for ex : Meter_ID=METER_005)

In [65]:
wwdata[wwdata["Meter_ID"] == "METER_005"]

,DateTime,Meter_ID,Flow_Rate,Pressure,Temperature,Water_Volume
4,2024-01-01 04:00:00,METER_005,1.34,2.78,15.1,420
9,2024-01-01 09:00:00,METER_005,1.67,2.95,15.4,530
14,2024-01-01 14:00:00,METER_005,1.89,3.05,15.6,600
19,2024-01-01 19:00:00,METER_005,2.12,3.18,15.8,680
24,2024-01-02 00:00:00,METER_005,1.23,2.88,15.2,410
29,2024-01-02 05:00:00,METER_005,1.45,2.82,15.2,450
34,2024-01-02 10:00:00,METER_005,2.01,3.12,15.7,640
39,2024-01-02 15:00:00,METER_005,2.23,3.28,15.9,710
44,2024-01-02 20:00:00,METER_005,1.67,2.98,15.5,530
49,2024-01-03 01:00:00,METER_005,1.34,2.85,15.1,420


5. Group data by meter and time periods

In [66]:
wwdata["DateTime"] = pd.to_datetime(wwdata["DateTime"])

meter_time_groups = wwdata.groupby(["Meter_ID", "DateTime"])

6. Calculate total, average, and peak consumption

    - **Total** = sum of water volume over the period
    - **Average** = mean water volume
    - **Peak** = max water volume

In [67]:
wwdata["Water_Volume"].agg(["sum", "mean", "max"])

sum     27020.000000
mean      491.272727
max      1020.000000
Name: Water_Volume, dtype: float64

7. Identify consumption trends

    Track how flow or volume changes over time for a meter.

In [68]:
meter = wwdata[wwdata["Meter_ID"] == "METER_001"].sort_values("DateTime")
meter[["DateTime", "Flow_Rate" ,"Water_Volume"]]

,DateTime,Flow_Rate,Water_Volume
0,2024-01-01 00:00:00,1.23,450
5,2024-01-01 05:00:00,1.45,480
10,2024-01-01 10:00:00,2.34,750
15,2024-01-01 15:00:00,2.45,780
20,2024-01-01 20:00:00,1.98,630
25,2024-01-02 01:00:00,1.56,510
30,2024-01-02 06:00:00,4.67,950
35,2024-01-02 11:00:00,5.12,1020
40,2024-01-02 16:00:00,4.89,980
45,2024-01-02 21:00:00,2.12,680


8. Compare meters for anomalies

Look for meters with unusually high or low flow, pressure, or water volume compared to others.

In [69]:
comparison = wwdata.groupby("Meter_ID")[["Flow_Rate", "Pressure", "Water_Volume"]].mean()
print("Flow Rate: \n", comparison.Flow_Rate.sort_values())
print("Pressure: \n", comparison.Pressure.sort_values())
print("Water Volume: \n", comparison.Water_Volume.sort_values())

Flow Rate: 
 Meter_ID
METER_004    0.635455
METER_002    0.991818
METER_005    1.692727
METER_003    1.903636
METER_001    2.660000
Name: Flow_Rate, dtype: float64
Pressure: 
 Meter_ID
METER_004    2.328182
METER_001    2.483636
METER_002    2.550000
METER_005    2.985455
METER_003    3.235455
Name: Pressure, dtype: float64
Water Volume: 
 Meter_ID
METER_004    235.454545
METER_002    365.454545
METER_005    538.181818
METER_003    616.363636
METER_001    700.909091
Name: Water_Volume, dtype: float64


#### More Cases :

9. What is the total water volume for each meter?

In [70]:
wwdata.groupby("Meter_ID")["Water_Volume"].sum()

Meter_ID
METER_001    7710
METER_002    4020
METER_003    6780
METER_004    2590
METER_005    5920
Name: Water_Volume, dtype: int64

10. Find readings where pressure is above the average pressure.

In [71]:
avg_Pressure = wwdata["Pressure"].mean()
wwdata.loc[wwdata.Pressure > avg_Pressure]

,DateTime,Meter_ID,Flow_Rate,Pressure,Temperature,Water_Volume
0,2024-01-01 00:00:00,METER_001,1.23,2.85,15.2,450
2,2024-01-01 02:00:00,METER_003,1.56,3.12,15.5,520
4,2024-01-01 04:00:00,METER_005,1.34,2.78,15.1,420
5,2024-01-01 05:00:00,METER_001,1.45,2.92,15.3,480
7,2024-01-01 07:00:00,METER_003,1.78,3.25,15.7,580
9,2024-01-01 09:00:00,METER_005,1.67,2.95,15.4,530
10,2024-01-01 10:00:00,METER_001,2.34,3.15,16.1,750
12,2024-01-01 12:00:00,METER_003,2.01,3.38,15.9,640
14,2024-01-01 14:00:00,METER_005,1.89,3.05,15.6,600
15,2024-01-01 15:00:00,METER_001,2.45,3.25,16.2,780


11. Create a dataset summary for the final report.

find the number of readings, the number of unique `Meter_ID` values, and the date range in `DateTime`.

In [72]:
print("Number of readings:", len(wwdata))
print("Number of meters:", wwdata["Meter_ID"].nunique())
print("Date range:", wwdata["DateTime"].min(), "to", wwdata["DateTime"].max())

Number of readings: 55
Number of meters: 5
Date range: 2024-01-01 00:00:00 to 2024-01-03 06:00:00


____________________________________________________________________________________________________

## Final Results Report

### Dataset Overview
- Number of readings: **55**.
- Number of `Meter_ID` values: **5**.
- `DateTime` range: **2024-01-01 00:00:00** to **2024-01-03 06:00:00**.

### Data Quality
- `DateTime` and `Meter_ID` have type `str`.
- `Flow_Rate`, `Pressure`, and `Temperature` have type `float64`; `Water_Volume` has type `int64`.
- Missing values: **0** in all columns.
- Duplicate readings: **0**.

### Outliers
- The IQR method identified three readings for `METER_001` as outliers in both `Flow_Rate` and `Pressure`.
- The outlier `DateTime` values are **2024-01-02 06:00:00**, **2024-01-02 11:00:00**, and **2024-01-02 16:00:00**.

### Questions and Results
- Highest `Flow_Rate`: `METER_001` with **5.12**.
- Average `Temperature` for `METER_003`: **15.78181818181818**.
- Total `Water_Volume` for `METER_005` on **2024-01-01**: **2230**.
- Total `Water_Volume`: **27020**.
- Average `Water_Volume`: **491.272727**.
- Peak `Water_Volume`: **1020**.

### Total `Water_Volume` by `Meter_ID`
| Meter_ID | Total Water_Volume |
| --- | ---: |
| METER_001 | 7710 |
| METER_002 | 4020 |
| METER_003 | 6780 |
| METER_004 | 2590 |
| METER_005 | 5920 |

### Average Values by `Meter_ID`
| Meter_ID | Flow_Rate | Pressure | Water_Volume |
| --- | ---: | ---: | ---: |
| METER_001 | 2.660000 | 2.483636 | 700.909091 |
| METER_002 | 0.991818 | 2.550000 | 365.454545 |
| METER_003 | 1.903636 | 3.235455 | 616.363636 |
| METER_004 | 0.635455 | 2.328182 | 235.454545 |
| METER_005 | 1.692727 | 2.985455 | 538.181818 |